# Gate field notes

My quick way to learn a gate is to pick an input where its effect shows up. For X I use |0⟩. For Z I use |+⟩ = (|0⟩ + |1⟩)/√2, because I want to see the two amplitudes change relative to each other.

In [1]:
from math import pi

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

## One wire

Read each array as [amplitude of 0, amplitude of 1]. Python writes the imaginary unit as `j`. A common phase on the whole array has no observable effect. A phase difference between entries can change a later measurement.

RX, RY and RZ rotate the qubit around the indicated axis of the Bloch sphere. From |0⟩, compare the second amplitude after RX and RY. U takes three angles and can describe any single-qubit gate up to global phase.

In [2]:
def show(name, gate, start="+"):
    circuit = QuantumCircuit(1)
    if start == "+":
        circuit.h(0)
    gate(circuit)
    state = Statevector.from_instruction(circuit)
    print(f"{name:12} on |{start}>   {state.data.round(3)}")


# Start in |0> so a flip is easy to see.
show("I", lambda q: q.id(0), "0")       # Leave the state alone.
show("X", lambda q: q.x(0), "0")        # Exchange |0> and |1>.
show("Y", lambda q: q.y(0), "0")        # |0> -> i|1>, |1> -> -i|0>.
show("H", lambda q: q.h(0), "0")        # Split the amplitude evenly.
show("SX", lambda q: q.sx(0), "0")      # Two SX gates make X.
show("SX dagger", lambda q: q.sxdg(0), "0")  # Undo SX.

# Start in |+> so both amplitudes are present. Watch the second one.
show("Z", lambda q: q.z(0))             # Change its sign.
show("S", lambda q: q.s(0))             # Multiply it by i.
show("S dagger", lambda q: q.sdg(0))    # Undo S.
show("T", lambda q: q.t(0))             # Add pi/4 of phase.
show("T dagger", lambda q: q.tdg(0))    # Undo T.
show("P(pi/3)", lambda q: q.p(pi/3, 0))  # Choose the phase angle.

# Angles are in radians. Try pi in place of pi/2.
show("RX(pi/2)", lambda q: q.rx(pi/2, 0), "0")
show("RY(pi/2)", lambda q: q.ry(pi/2, 0), "0")
show("RZ(pi/2)", lambda q: q.rz(pi/2, 0))
show("U", lambda q: q.u(pi/2, 0, pi, 0), "0")  # These angles give H.

I            on |0>   [1.+0.j 0.+0.j]
X            on |0>   [0.+0.j 1.+0.j]
Y            on |0>   [0.+0.j 0.+1.j]
H            on |0>   [0.707+0.j 0.707+0.j]
SX           on |0>   [0.5+0.5j 0.5-0.5j]
SX dagger    on |0>   [0.5-0.5j 0.5+0.5j]
Z            on |+>   [ 0.707+0.j -0.707+0.j]
S            on |+>   [0.707+0.j    0.   +0.707j]
S dagger     on |+>   [0.707+0.j    0.   -0.707j]
T            on |+>   [0.707+0.j  0.5  +0.5j]
T dagger     on |+>   [0.707+0.j  0.5  -0.5j]
P(pi/3)      on |+>   [0.707+0.j    0.354+0.612j]
RX(pi/2)     on |0>   [0.707+0.j    0.   -0.707j]
RY(pi/2)     on |0>   [0.707+0.j 0.707+0.j]
RZ(pi/2)     on |+>   [0.5-0.5j 0.5+0.5j]
U            on |0>   [0.707+0.j 0.707+0.j]


## Getting a phase into the readout

Z turns |+⟩ into |−⟩. Both give 50/50 in the usual measurement basis. H can tell them apart because it sends |+⟩ to |0⟩ and |−⟩ to |1⟩.

In [3]:
probe = QuantumCircuit(1)
probe.h(0)
probe.z(0)
print("Before final H:", Statevector(probe).probabilities().round(3))
probe.h(0)
print("After final H: ", Statevector(probe).probabilities().round(3))

Before final H: [0.5 0.5]
After final H:  [0. 1.]


## Controls and swaps

For CX, CY and CH the first argument is the control. A control of 1 applies X, Y or H to the target. CZ gives |11⟩ a minus sign, and controlled phase gives that branch a chosen phase. SWAP exchanges two wires.

Toffoli flips the last wire if both controls are 1. Fredkin swaps the last two wires if the first is 1. Trace the input 011 by writing q0, q1 and q2 on paper first.

In [4]:
def mapping(label, gate):
    circuit = QuantumCircuit(len(label))
    gate(circuit)
    result = Statevector.from_label(label).evolve(circuit)
    return {str(k): complex(v) for k, v in result.to_dict().items()}


for label in ("00", "01", "10", "11"):
    print(label, "CX", mapping(label, lambda q: q.cx(0, 1)),
          "CZ", mapping(label, lambda q: q.cz(0, 1)),
          "SWAP", mapping(label, lambda q: q.swap(0, 1)))

print("CY:", mapping("01", lambda q: q.cy(0, 1)))
print("CH:", mapping("01", lambda q: q.ch(0, 1)))
print("Controlled phase:", mapping("11", lambda q: q.cp(pi/2, 0, 1)))
print("Toffoli:", mapping("011", lambda q: q.ccx(0, 1, 2)))
print("Fredkin:", mapping("011", lambda q: q.cswap(0, 1, 2)))

00 CX {'00': (1+0j)} CZ {'00': (1+0j)} SWAP {'00': (1+0j)}
01 CX {'11': (1+0j)} CZ {'01': (1+0j)} SWAP {'10': (1+0j)}
10 CX {'10': (1+0j)} CZ {'10': (1+0j)} SWAP {'01': (1+0j)}
11 CX {'01': (1+0j)} CZ {'11': (-1+0j)} SWAP {'11': (1+0j)}
CY: {'11': 1j}
CH: {'01': (0.7071067811865475+0j), '11': (0.7071067811865475+0j)}
Controlled phase: {'11': (6.123233995736766e-17+1j)}
Toffoli: {'111': (1+0j)}
Fredkin: {'101': (1+0j)}


Try changing the three-wire input to 010. Which gate still acts? Check your answer by reading the rightmost bit first. For this input both circuits leave the state alone.